# Activation Functions

> Based on Stanford CS230 — C1M3, Andrew Ng / DeepLearning.AI

The activation function $g(z)$ introduces **non-linearity** into the network. Without it, any depth of linear layers collapses to a single linear transformation — the network could not learn complex patterns.

## Learning Objectives

1. State the formula and derivative for sigmoid, tanh, ReLU, and Leaky ReLU
2. Explain the vanishing gradient problem and how ReLU addresses it
3. Select the right activation for hidden vs output layers
4. Understand why linear activations kill the network's expressiveness

## Summary Table

| Activation | Formula | Range | Derivative | Notes |
|---|---|---|---|---|
| **Sigmoid** | $\sigma(z) = \frac{1}{1+e^{-z}}$ | $(0,1)$ | $\sigma(1-\sigma)$ | Output layer (binary), vanishes for large $|z|$ |
| **Tanh** | $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$ | $(-1,1)$ | $1 - \tanh^2$ | Zero-centred; often beats sigmoid in hidden layers |
| **ReLU** | $\max(0, z)$ | $[0,\infty)$ | $\mathbf{1}[z>0]$ | Default for hidden layers; trains 10× faster |
| **Leaky ReLU** | $\max(\alpha z, z),\;\alpha=0.01$ | $(-\infty,\infty)$ | $1$ or $\alpha$ | Fixes "dying ReLU" for $z<0$ |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

z = np.linspace(-4, 4, 400)
alpha = 0.1  # leaky ReLU slope

activations = {
    'Sigmoid':    (lambda z: 1/(1+np.exp(-z)),        lambda z: (s:=1/(1+np.exp(-z))) * (1-s)),
    'Tanh':       (np.tanh,                            lambda z: 1 - np.tanh(z)**2),
    'ReLU':       (lambda z: np.maximum(0, z),         lambda z: (z > 0).astype(float)),
    'Leaky ReLU': (lambda z: np.where(z>0, z, alpha*z), lambda z: np.where(z>0, 1.0, alpha)),
}

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('Activation Functions and Their Derivatives', fontsize=14, fontweight='bold')

for col, (name, (fn, dfn)) in enumerate(activations.items()):
    # Top row: function
    ax = axes[0][col]
    ax.plot(z, fn(z), color=P[col], lw=2.5)
    ax.axhline(0, color='#bbb', lw=0.8); ax.axvline(0, color='#bbb', lw=0.8)
    ax.set_title(name)
    ax.set_xlabel('z')
    ax.set_ylabel('g(z)' if col == 0 else '')
    ax.grid(True)

    # Bottom row: derivative
    ax = axes[1][col]
    ax.plot(z, dfn(z), color=P[col], lw=2.5, ls='--')
    ax.axhline(0, color='#bbb', lw=0.8); ax.axvline(0, color='#bbb', lw=0.8)
    ax.set_title(f"g'(z)  — {name}")
    ax.set_xlabel('z')
    ax.set_ylabel("g'(z)" if col == 0 else '')
    ax.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# --- Vanishing gradient illustration ---
# Gradient of sigmoid shrinks as |z| grows.
# After L layers of sigmoid, the gradient shrinks exponentially.

L_values = range(1, 30)
z_val = 2.0  # z far from 0 — sigmoid gradient ≈ 0.1
s = 1/(1+np.exp(-z_val))
sig_grad = s*(1-s)   # ≈ 0.10 at z=2

sigmoid_chain = [sig_grad ** L for L in L_values]
relu_chain    = [1.0           for L in L_values]  # ReLU grad = 1 for z>0

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.semilogy(list(L_values), sigmoid_chain, color=P[1], lw=2.5, label=f'Sigmoid  (g\'≈{sig_grad:.2f} per layer)')
ax.semilogy(list(L_values), relu_chain,    color=P[3], lw=2.5, ls='--', label='ReLU  (g\'=1 for z>0)')
ax.set_xlabel('Number of layers L')
ax.set_ylabel('Cumulative gradient magnitude (log scale)')
ax.set_title('Vanishing Gradient: Sigmoid vs ReLU')
ax.legend(fontsize=10)
ax.grid(True)
ax.axhline(1e-5, color='#aaa', lw=0.8, ls=':')
ax.text(22, 2e-5, 'effectively zero', color='#aaa', fontsize=9)
plt.tight_layout()
plt.show()

print("Sigmoid gradient after 20 layers:", sigmoid_chain[19])
print("ReLU    gradient after 20 layers:", relu_chain[19])


## Choosing an Activation Function

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 580 200" width="580" height="200" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <!-- title -->
  <text x="290" y="22" text-anchor="middle" fill="#333" font-size="12" font-weight="bold">Which activation to use?</text>
  <!-- Decision tree nodes -->
  <!-- Root: is it output layer? -->
  <rect x="200" y="35" width="180" height="36" rx="6" fill="#5b9bd5" fill-opacity="0.15" stroke="#5b9bd5" stroke-width="1.5"/>
  <text x="290" y="55" text-anchor="middle" fill="#3a7bbf" font-size="11" font-weight="bold">Output layer?</text>
  <!-- Yes branch → binary? -->
  <rect x="60"  y="110" width="160" height="36" rx="6" fill="#e05c5c" fill-opacity="0.12" stroke="#e05c5c" stroke-width="1.5"/>
  <text x="140" y="130" text-anchor="middle" fill="#b03a3a" font-size="11" font-weight="bold">Binary classification?</text>
  <!-- No branch → Hidden → use ReLU -->
  <rect x="360" y="110" width="160" height="36" rx="6" fill="#7ecba1" fill-opacity="0.12" stroke="#7ecba1" stroke-width="1.5"/>
  <text x="440" y="130" text-anchor="middle" fill="#3d7a5e" font-size="11" font-weight="bold">Hidden layer → ReLU</text>
  <!-- Binary Yes → Sigmoid -->
  <rect x="0"  y="175" width="100" height="18" rx="4" fill="#f4b942" fill-opacity="0.2" stroke="#f4b942" stroke-width="1.2"/>
  <text x="50" y="188" text-anchor="middle" fill="#a07010" font-size="10" font-weight="bold">→ Sigmoid</text>
  <!-- Binary No → Linear or Softmax -->
  <rect x="130" y="175" width="120" height="18" rx="4" fill="#f4b942" fill-opacity="0.2" stroke="#f4b942" stroke-width="1.2"/>
  <text x="190" y="188" text-anchor="middle" fill="#a07010" font-size="10" font-weight="bold">→ Linear / Softmax</text>
  <!-- Arrows -->
  <line x1="200" y1="53" x2="140" y2="110" stroke="#888" stroke-width="1.3"/>
  <text x="155" y="90" fill="#666" font-size="10">Yes</text>
  <line x1="380" y1="53" x2="440" y2="110" stroke="#888" stroke-width="1.3"/>
  <text x="400" y="90" fill="#666" font-size="10">No</text>
  <line x1="100" y1="146" x2="50"  y2="175" stroke="#888" stroke-width="1.3"/>
  <text x="55"  y="167" fill="#666" font-size="10">Yes</text>
  <line x1="180" y1="146" x2="190" y2="175" stroke="#888" stroke-width="1.3"/>
  <text x="182" y="167" fill="#666" font-size="10">No</text>
</svg>

> **Practical rule:** always start with **ReLU** for hidden layers. Only switch to leaky ReLU if you observe many "dead" neurons (activations stuck at 0) during training.
